# Pixhawk 2.4.8 四軸機飛控入門
共 20 個 Python 儲存格。請從「執行階段 → 全部執行」開始。第 18 格預覽，第 20 格下載。資料為已完成取材的 Wiki 快照。

## 01　準備執行環境

In [ ]:
from html import escape
from pathlib import Path
from IPython.display import HTML, display
from base64 import b64encode
import json, re
from zipfile import ZipFile, ZIP_DEFLATED
from urllib.parse import urlparse
folder = Path("Pixhawk248_output")
folder.mkdir(exist_ok=True)

## 02　載入六篇已整理的來源筆記

In [ ]:
wiki = [{'id': 'W01',
  'topic': '辨認板子',
  'question': '2.4.8 這串數字代表什麼？',
  'idea': 'RadioLink 在硬體比較表中用 2.4.8 標示自家的 PIXHAWK。看到這個數字時，第一步是記下廠牌與板上標示，再找對應手冊。',
  'example': '同學只給你一張寫著「2.4.8」的外殼照片。你可以記錄型號，但要回答接頭腳位，還需要廠牌與對應版本的文件。',
  'url': 'https://www.radiolink.com/en/comparison_of_pixhawk_series',
  'source': 'RadioLink｜Comparison of PIXHAWK Series',
  'points': ['讀板上標示', '確認製造商', '找同版手冊'],
  'summary': '只知道 2.4.8 時，先記錄廠牌，再查接頭定義。'},
 {'id': 'W02',
  'topic': '讀懂名稱',
  'question': 'Pixhawk 1、FMUv2 和 PX4 各指什麼？',
  'idea': 'Pixhawk 1 是飛控產品名稱，FMUv2 是它採用的參考硬體設計；PX4 則是可以在支援硬體上執行的飛控軟體。',
  'example': '閱讀文件時先看它在描述哪一層。找電路設計看硬體文件；想知道軟體如何算出控制指令，就讀 PX4 架構。',
  'url': 'https://github.com/pixhawk/Hardware/blob/master/README.md',
  'source': 'Pixhawk｜Hardware · FMU Versions',
  'points': ['產品名稱', '參考硬體設計', '飛控軟體'],
  'summary': '查資料前，先判斷問題屬於硬體還是軟體。'},
 {'id': 'W03',
  'topic': '理解控制',
  'question': '一陣風吹斜四軸機，飛控怎麼知道要修正？',
  'idea': '感測器提供量測值，估測器據此推算機身姿態；控制器比較目標與估測值，產生修正指令。下一次量測會反映修正後的狀態。',
  'example': '假設目標是保持水平。風吹後，估測姿態向右傾，控制器依偏差要求修正；每次更新姿態，都會重新計算還需要多少修正。',
  'url': 'https://docs.px4.io/main/en/concept/architecture',
  'source': 'PX4｜Architectural Overview',
  'points': ['量測姿態相關訊號', '估測機身姿態', '依目標偏差修正'],
  'summary': '每一次修正後再量測，這就是回授。'},
 {'id': 'W04',
  'topic': '認識周邊',
  'question': '同一顆 GPS／羅盤模組，為什麼有兩種資料？',
  'idea': 'GPS 提供衛星定位資料，羅盤量測磁場，協助估算機頭朝向。經典 3DR Pixhawk 文件的組合模組分別使用 GPS 與 I²C 介面。',
  'example': '把機身留在桌上原地轉向：位置幾乎一樣，機頭卻已改變方向。這說明位置與朝向是兩個不同的問題。此處的介面是參考機案例。',
  'url': 'https://docs.px4.io/main/en/assembly/quick_start_pixhawk',
  'source': 'PX4｜Pixhawk Wiring Quick Start · GPS + Compass',
  'points': ['GPS 回答位置', '羅盤協助估算航向', '查模組與主板介面'],
  'summary': '機身的位置和機頭的方向可以分別改變。'},
 {'id': 'W05',
  'topic': '判讀校正',
  'question': '為什麼校正羅盤時要轉動機身？',
  'idea': '磁力計量測磁場。QGroundControl 會提示使用者改變姿態，再繞指定軸旋轉，讓系統取得不同方向的磁場讀值。',
  'example': '若把羅盤放在磁場異常的地方，觀察到的方向也可能受到影響。判讀羅盤問題時，應把量測環境一起記錄。',
  'url': 'https://docs.px4.io/main/en/config/compass',
  'source': 'PX4｜Compass Calibration',
  'points': ['讀取磁場', '依提示轉動', '留意環境干擾'],
  'summary': '校正羅盤要取得多個方向的磁場讀值。'},
 {'id': 'W06',
  'topic': '比較感測器',
  'question': '為什麼加速度計校正時反而要放穩？',
  'idea': '在不同姿態下靜置機身，重力在感測器各軸上的分量會改變。校正需要各姿態穩定的讀值，QGroundControl 會提示何時換面。',
  'example': '若畫面提示保持靜止，持續轉動會破壞這次量測的條件。可以用「羅盤轉動、加速度計分面靜置」記住兩種校正動作。',
  'url': 'https://docs.px4.io/main/en/config/accelerometer',
  'source': 'PX4｜Accelerometer Calibration',
  'points': ['依提示換面', '每面保持靜止', '等待取樣完成'],
  'summary': '校正加速度計時，每一面都要有穩定的量測。'}]


## 03　檢查來源與版本標記

In [ ]:
ids = [w["id"] for w in wiki]
assert len(set(ids)) == len(wiki) >= 5
assert len({w["url"] for w in wiki}) == len(wiki)
for w in wiki:
    assert urlparse(w["url"]).scheme == "https" and urlparse(w["url"]).hostname
print("來源編號與網址格式通過", ids)

## 04　核對 Wiki 必填欄位

In [ ]:
for w in wiki:
    for key in ("topic", "question", "idea", "example", "summary", "source"):
        assert w[key].strip(), (w["id"], key)
    assert len(w["points"]) == 3
print("六個條目皆有摘要、例子與三個知識點")

## 05　建立可查詢的 Wiki 索引

In [ ]:
index = {w["id"]: w for w in wiki}
def search_wiki(term):
    return [w["id"] for w in wiki if term in " ".join([w["question"], w["idea"], w["example"]])]
print("查詢「羅盤」", search_wiki("羅盤"))

## 06　儲存 Wiki 知識庫

In [ ]:
wiki_file = folder / "wiki.json"
wiki_file.write_text(json.dumps(wiki, ensure_ascii=False, indent=2), encoding="utf-8")

## 07　依學習順序取回條目

In [ ]:
order = ["W01","W02","W03","W04","W05","W06"]
lessons = [index[key] for key in order]
print("\n".join(w["question"] for w in lessons))

## 08　設定捲動式簡報外觀

In [ ]:
css = "*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;background:#faf8ef;color:#222e28;font:17px/1.75 system-ui,'Microsoft JhengHei',sans-serif}main,header,nav>div{max-width:1040px;margin:auto;padding:24px}header{display:grid;grid-template-columns:1.15fr 1fr;align-items:center;min-height:380px;gap:24px}h1{font-size:clamp(42px,6vw,76px);line-height:1.08;letter-spacing:-.055em;margin:10px 0}h2{font-size:clamp(24px,3vw,36px);line-height:1.35}small{font-weight:700;letter-spacing:.12em;color:#336453}nav{background:#203d33;position:sticky;top:0;z-index:2}nav>div{display:flex;gap:20px;padding-block:10px;flex-wrap:wrap}nav a{color:white}a{color:#195742;text-underline-offset:4px}section,article,details{scroll-margin-top:90px}article{padding:32px 0;border-top:1px solid #9cafa2}aside,details{padding:16px 20px;background:#e7ecdf;border-radius:8px;margin:12px 0}summary{cursor:pointer;font-weight:700}figure{margin:0;text-align:center}svg{width:100%;max-height:280px}figcaption{font-size:14px;color:#486353}li{margin:.25em 0}@media(max-width:630px){header{grid-template-columns:1fr}header svg{height:170px}nav>div{gap:12px;font-size:14px}}"

## 09　繪製四軸機俯視圖

In [ ]:
diagram = '<figure><svg viewBox="0 0 340 230" role="img" aria-label="四軸機俯視示意，中央為飛控板"><g fill="none" stroke="#75988b" stroke-width="9"><path d="m65 44 210 142m0-142L65 186"/></g><g transform="translate(65 44) rotate(28)"><circle r="36" fill="#e1eadf" stroke="#8ba598"/><ellipse rx="33" ry="7" fill="#365c4f"/><circle r="5" fill="#d2ad47"/></g><g transform="translate(275 44) rotate(-28)"><circle r="36" fill="#e1eadf" stroke="#8ba598"/><ellipse rx="33" ry="7" fill="#365c4f"/><circle r="5" fill="#d2ad47"/></g><g transform="translate(65 186) rotate(-28)"><circle r="36" fill="#e1eadf" stroke="#8ba598"/><ellipse rx="33" ry="7" fill="#365c4f"/><circle r="5" fill="#d2ad47"/></g><g transform="translate(275 186) rotate(28)"><circle r="36" fill="#e1eadf" stroke="#8ba598"/><ellipse rx="33" ry="7" fill="#365c4f"/><circle r="5" fill="#d2ad47"/></g><rect x="138" y="78" width="64" height="78" rx="12" fill="#214336"/><path d="M159 106l11-12 11 12" fill="none" stroke="#edc85c" stroke-width="3"/><text x="170" y="135" fill="white" text-anchor="middle" font-size="14">飛控</text></svg><figcaption>四軸機俯視示意 · 箭頭指向機頭</figcaption></figure>'

## 10　整理製作流程與反思

In [ ]:
run = '<h2>先討論讀者與範圍</h2><p>原本考慮以 6C 為例，討論後依指定改為 2.4.8。讀者設定為第一次接觸飛控的同學，目標是能辨認板型、解釋一次姿態修正，並分辨兩種校正。產品案例採 RadioLink，經典架構以 Pixhawk 1 為參考。</p><h2>資料與生成迴圈</h2><ol><li>用教學問題找原廠或專案文件；頁面失效或只有導覽時，退回搜尋。</li><li>逐篇確認適用板型或軟體層級，再整理成包含問題、摘要、情境與來源的 Wiki 條目。</li><li>依學習順序查詢 W01–W06，每張投影片選一個問題，以條目生成解說與例子。</li><li>核對同頁的 Wiki 連結與原文。遇到硬體細節缺乏依據，回條目縮小適用範圍，再生成受影響的頁面。</li></ol><h2>一次實際執行</h2><p>取材時有兩個候選頁無法使用。PX4 的 concept/flight_stack.html 回傳 404，改查 Architectural Overview；flight_controller/pixhawk 雖能開啟，正文只有導覽與編輯連結，改採 Pixhawk 專案的 Hardware 文件。最後收錄六篇，每一篇對應一個 Wiki 條目與一張主題投影片。第七頁用兩道題整合 W04–W06。</p><h2>反思與調整</h2><p>這次觀察到最多問題的是「取得可用來源」，共有兩個候選頁需要替換。只看網址或頁名會把空頁帶入摘要。因此將「讀正文、定位支持句、確認適用板型」移到寫 Wiki 之前。這樣來源不合適時只需重新取材，後面的投影片也較不容易沿用錯誤型號。</p>'

## 11　定義每章教學版型

In [ ]:
def lesson(w):
    points = ''.join(f'<li>{escape(p)}</li>' for p in w['points'])
    return f"""<article><small>{w['id']} ／ {w['topic']}</small><h2>{w['question']}</h2><p>{w['idea']}</p><strong>三個重要知識點</strong><ul>{points}</ul><aside><b>情境練習</b><br>{w['example']}</aside><p><b>小結　</b>{w['summary']}</p><a href="#{w['id']}">查看 Wiki 依據 ↗</a></article>"""

## 12　產生六個教學章節

In [ ]:
cards = ''.join(lesson(w) for w in lessons)

## 13　建立可展開的 Wiki 來源區

In [ ]:
sources = ''.join(f"""<div id="{w['id']}"><details><summary>{w['id']}　{w['topic']}</summary><p>{w['idea']}</p><a href="{w['url']}" target="_blank" rel="noopener">{w['source']} ↗</a></details></div>""" for w in wiki)

## 14　組合互動式教學網頁

In [ ]:
page = f"""<!doctype html><html lang="zh-Hant"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Pixhawk 2.4.8 飛控入門</title><style>{css}</style></head><body><header><div><small>四軸機飛控入門</small><h1>Pixhawk<br>2.4.8</h1><p>風吹斜了四軸機，它怎麼知道要回正？從機身中央的飛控板開始，沿著感測資料找答案。</p></div>{diagram}</header><nav><div><a href="#lesson">飛控教學</a><a href="#quiz">自我檢查</a><a href="#wiki">Wiki 來源</a><a href="#log">製作紀錄</a></div></nav><main><section id="lesson">{cards}</section><section id="quiz"><h2>自我檢查</h2><details><summary>機身在桌上原地轉向，哪一筆資料更能顯示變化？</summary><p>羅盤航向。機頭方向已變，GPS 位置幾乎相同。</p></details><details><summary>校正時，哪個感測器需要換面後保持靜止？</summary><p>加速度計。各面需要穩定讀值；羅盤校正才依提示轉動機身。</p></details></section><section id="wiki"><h2>LLM Wiki 知識庫</h2>{sources}</section><section id="log"><h2>製作紀錄與反思</h2>{run}</section></main></body></html>"""

## 15　檢查章節、超連結與自我測驗

In [ ]:
targets = re.findall(r' id="([^"]+)"', page)
links = re.findall(r'href="#([^"]+)"', page)
assert len(targets) == len(set(targets))
assert set(links) <= set(targets)
assert page.count("<article>") == len(lessons)
assert all(w["url"] in page for w in lessons)
assert page.split('id="quiz"',1)[1].split("</section>",1)[0].count("<details>") == 2
print("章節、頁內連結、六個來源與兩道測驗皆通過")

## 16　匯出獨立 HTML

In [ ]:
output = folder / "index.html"
output.write_text(page, encoding="utf-8")
print(output.resolve())

## 17　寫入本次檢查結果

In [ ]:
from datetime import datetime, timezone
report = {"執行時間_UTC":datetime.now(timezone.utc).isoformat(),"教學順序":order,"來源數":len(wiki),"章節數":len(lessons),"已驗證的頁內連結數":len(links),"自我測驗題數":2,"HTML位元組":output.stat().st_size,"檢查結果":"通過","檢查範圍":"本次重新檢查保存資料及產生的 HTML。原始網站取材紀錄見網頁製作紀錄。"}
log_file = folder / "checks.json"
log_file.write_text(json.dumps(report, ensure_ascii=False, indent=2),encoding="utf-8")
print(report)

## 18　在 Colab 預覽教學頁

In [ ]:
encoded = b64encode(page.encode()).decode()
display(HTML(f'<iframe title="Pixhawk 飛控教學" src="data:text/html;charset=utf-8;base64,{encoded}" style="width:100%;height:740px;border:1px solid #abbbae"></iframe>'))

## 19　打包本次產生的成果

In [ ]:
archive = Path("Pixhawk248-results.zip")
with ZipFile(archive,"w",ZIP_DEFLATED) as z:
    for artifact in (output,wiki_file,log_file):
        z.write(artifact,artifact.name)
print(archive.name)

## 20　下載本次成果 ZIP

In [ ]:
try:
    from google.colab import files
except ImportError:
    print("成果位置", archive.resolve())
else:
    files.download(str(archive))